# Assignment 3
#### Name - Nitish Kumar
#### Roll No - 231110033
#### Dept. - M.Tech(C.S.E)
## For the ChatGPT

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Uploading all the datafiles

In [8]:
with open("test.en","r",encoding="utf-8") as file:
    eng_sentences=file.readlines()

eng_sentences=[sentence for sentence in eng_sentences]


In [7]:
with open("test.hi","r",encoding="utf-8") as file:
    hin_sentences=file.readlines()

hin_sentences=[sentence for sentence in hin_sentences]


In [9]:
with open("test.gu","r",encoding="utf-8") as file:
    guj_sentences=file.readlines()

guj_sentences=[sentence for sentence in guj_sentences]


### Sampling random 50 sentences from Hindi, English and Gujrati Data

In [10]:
import random
random.seed(1)
sample_indices = random.sample(range(len(eng_sentences)), 50)
eng_sample = [eng_sentences[i] for i in sample_indices]
hin_sample = [hin_sentences[i] for i in sample_indices]
guj_sample = [guj_sentences[i] for i in sample_indices]

### Saving those 50 sentences in sample files
#### So that I can use chatGPT to translate only those specific 50 sentences

In [11]:
# Save English sample to a text file
with open('/content/eng_sample.txt', 'w', encoding='utf-8') as file:
    for line in eng_sample:
        file.write(line)

# Save Hindi sample to a text file
with open('/content/hin_sample.txt', 'w', encoding='utf-8') as file:
    for line in hin_sample:
        file.write(line)

# Save Gujarati sample to a text file
with open('/content/guj_sample.txt', 'w', encoding='utf-8') as file:
    for line in guj_sample:
        file.write(line)


# Evaluation of models

### Calculation of BLEU Score

In [13]:
from nltk.translate.bleu_score import corpus_bleu

# Function to read text file content into a list
def read_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [line.strip() for line in file.readlines()]

# Function to calculate BLEU scores
def calculate_bleu_scores(references, translations):
    # Wrapping each reference sentence in a list because corpus_bleu expects a list of lists
    formatted_references = [[ref] for ref in references]
    bleu_score = corpus_bleu(formatted_references, translations)
    return bleu_score

def print_bleu_scores(model_name, references, translations):
    bleu_score = calculate_bleu_scores(references, translations)
    print(f"\nBLEU Score for {model_name}: {bleu_score:.4f}")

# Paths to the files containing the reference and translated texts
hin_sample_file_path = 'hin_sample.txt'
eng_sample_file_path = 'eng_sample.txt'
guj_sample_file_path = 'guj_sample.txt'
chatgpt_eng_to_hi_file_path = 'chatgpt_eng_to_hi.txt'
chatgpt_hi_to_eng_file_path = 'chatgpt_hi_to_eng.txt'
chatgpt_hi_to_guj_file_path = 'chatgpt_hi_to_guj.txt'
chatgpt_guj_to_hi_file_path = 'chatgpt_guj_to_hi.txt'


# Reading contents of the files
sample_hindi = read_file(hin_sample_file_path)
translated_eng_to_hin = read_file(chatgpt_eng_to_hi_file_path)
sample_english = read_file(eng_sample_file_path)
translated_hin_to_eng = read_file(chatgpt_hi_to_eng_file_path)
sample_gujrati = read_file(guj_sample_file_path)
translated_hin_to_guj = read_file(chatgpt_hi_to_guj_file_path)
translated_guj_to_hin = read_file(chatgpt_guj_to_hi_file_path)

# Calculate and print BLEU scores for the translation of each model
print_bleu_scores("English to Hindi", sample_hindi, translated_eng_to_hin)
print_bleu_scores("Hindi to English", sample_english, translated_hin_to_eng)
print_bleu_scores("Hindi to Gujarati", sample_gujrati, translated_hin_to_guj)
print_bleu_scores("Gujarati to Hindi", sample_hindi, translated_guj_to_hin)



BLEU Score for English to Hindi: 0.7536

BLEU Score for Hindi to English: 0.6986

BLEU Score for Hindi to Gujarati: 0.6241

BLEU Score for Gujarati to Hindi: 0.7282


In [24]:
from nltk.translate.bleu_score import sentence_bleu

# Function to calculate sentence-level BLEU scores and return the average
def calculate_average_sentence_bleu(references, translations):
    score=0
    for ref, trans in zip(references, translations):
        score += sentence_bleu([ref], trans)
    # Calculate the average of the sentence BLEU scores
    average_score = score / 50 #if scores else 0
    return average_score

def print_average_sentence_bleu(model_name, references, translations):
    average_score = calculate_average_sentence_bleu(references, translations)
    print(f"\nAverage Sentence-level BLEU Score for {model_name}: {average_score:.4f}")

# Printing Average Sentence-level BLEU Score for each translation Model
print_average_sentence_bleu("English to Hindi", sample_hindi, translated_eng_to_hin)
print_average_sentence_bleu("Hindi to English", sample_english, translated_hin_to_eng)
print_average_sentence_bleu("Hindi to Gujarati", sample_gujrati, translated_hin_to_guj)
print_average_sentence_bleu("Gujarati to Hindi", sample_hindi, translated_guj_to_hin)



Average Sentence-level BLEU Score for English to Hindi: 0.7255

Average Sentence-level BLEU Score for Hindi to English: 0.6403

Average Sentence-level BLEU Score for Hindi to Gujarati: 0.6046

Average Sentence-level BLEU Score for Gujarati to Hindi: 0.7181


### Calculation of ROGUE Score

In [14]:
!pip install rouge

In [15]:
import numpy as np
from rouge import Rouge

# Initialize the Rouge scorer
rouge = Rouge()

def average_rouge_scores(translated_texts, reference_texts):
    """
    Args:
        translated_texts (list of str): The translated sentences.
        reference_texts (list of str): The ground truth reference sentences.

    Returns:
        dict: A dictionary containing averaged scores for ROUGE-1, ROUGE-2, and ROUGE-L.
    """
    scores = rouge.get_scores(translated_texts, reference_texts, avg=True)  # Use avg=True to get the average score
    return scores

def print_rouge_scores(model_name, scores):
    """

    Args:
        model_name (str): Description of the model translation direction.
        scores (dict): A dictionary containing averaged ROUGE scores.
    """
    print(f"\nROUGE Scores for {model_name} Translation:")
    print(f"{'Metric':<8} | {'Precision':<10} | {'Recall':<10} | {'F1-score':<10}")
    print('-' * 42)
    for key in ['rouge-1', 'rouge-2', 'rouge-l']:
        metric_name = key.upper().replace('ROUGE-', 'ROUGE ')
        precision = scores[key]['p']
        recall = scores[key]['r']
        f1_score = scores[key]['f']
        print(f"{metric_name:<8} | {precision:<10.4f} | {recall:<10.4f} | {f1_score:<10.4f}")

# Printing ROUGE Scores for each translation model
rouge_score_eng_to_hin = average_rouge_scores(translated_eng_to_hin, sample_hindi)
print_rouge_scores("English to Hindi", rouge_score_eng_to_hin)

rouge_score_hin_to_eng = average_rouge_scores(translated_hin_to_eng, sample_english)
print_rouge_scores("Hindi to English", rouge_score_hin_to_eng)

rouge_score_hin_to_guj = average_rouge_scores(translated_hin_to_guj, sample_gujrati)
print_rouge_scores("Hindi to Gujarati", rouge_score_hin_to_guj)

rouge_score_guj_to_hin = average_rouge_scores(translated_guj_to_hin, sample_hindi)
print_rouge_scores("Gujarati to Hindi", rouge_score_guj_to_hin)



ROUGE Scores for English to Hindi Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  | 0.6920     | 0.6904     | 0.6878    
ROUGE 2  | 0.4955     | 0.4932     | 0.4918    
ROUGE L  | 0.6655     | 0.6630     | 0.6610    

ROUGE Scores for Hindi to English Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  | 0.5982     | 0.5960     | 0.5919    
ROUGE 2  | 0.3727     | 0.3747     | 0.3697    
ROUGE L  | 0.5714     | 0.5662     | 0.5640    

ROUGE Scores for Hindi to Gujarati Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  | 0.5449     | 0.5210     | 0.5302    
ROUGE 2  | 0.2719     | 0.2614     | 0.2650    
ROUGE L  | 0.5152     | 0.4931     | 0.5017    

ROUGE Scores for Gujarati to Hindi Translation:
Metric   | Precision  | Recall     | F1-score  
------------------------------------------
ROUGE 1  |